# Split state model experiment

A split state model is one which separates each $-1/1$ spin into two negatively-connected $0/1$ spins. This separates the relation effects between the two states, such that the interaction effect on another spin may vary depending on whether the influencing spin is a $-1$ or a $+1$.

In [ ]:
from pathlib import Path

import numpy as np
from scipy.optimize import minimize
from sklearn.linear_model import LogisticRegression

from climate_attitudes.dataset import Dataset
from climate_attitudes.settings import Config
from climate_attitudes.visualisation import configure_mpl

configure_mpl(Path("../fonts/"))

np.set_printoptions(linewidth=200)

RANDOM_SEED = 202606031418

In [ ]:
config = Config(_env_file="../.env")
dataset = Dataset.load(
    config,
    name="reduced_no_imputation",
    with_imputation=False,
    verbose=False,
)
_, Y_bin, X = dataset.indices_to_numpy(
    kind="time-series", binarise=True, seed=RANDOM_SEED
)
_, Y_tern, X = dataset.indices_to_numpy(
    kind="time-series", ternarise=True, epsilon=0.15, seed=RANDOM_SEED
)

Y_bin[:, :, 5] *= -1
Y_tern[:, :, 5] *= -1

Consider the variables 'Belief in climate change' (index 0) and 'Support for climate policies' (index 7). 

If an individual does not believe in climate change, logically they should not support climate policy. However, individuals who do believe in climate change may nonetheless oppose climate policy for other reasons (e.g., cost or priority). Thus we expect the magnitude of the interaction effect from _lack of_ belief in climate change to _opposition_ toward climate policy to be **larger** than that from _belief_ in climate change to _support_ for climate policy.

We first fit the regular (non split-state) model by fitting a logistic regression for each variable on the prior timestep of the previous one. This serves as our baseline.

In [ ]:
res = LogisticRegression().fit(Y_bin[:, 0], Y_bin[:, 1, 7])
h_baseline = res.intercept_ / 2
J_baseline = res.coef_[0] / 2

In [ ]:
dataset.schema.get_short_names(kind="measurement")

In [ ]:
h_baseline

In [ ]:
J_baseline

Now separate $+1$ and $-1$ responses for each spin into two separate spins, and recode as $0$ and $1$. Fit logistic regressions for each of the new 'climate policy' spins, regressing on all prior spin states.

In [ ]:
Y_off = (Y_tern == -1).astype(np.int64)
Y_on = (Y_tern == 1).astype(np.int64)
Y_split = np.dstack((Y_off, Y_on))

In [ ]:
def nll_bin(params, X, y):
    J_i = params[:-1]
    h_i = params[-1]

    # Calculate effective local field for each individual
    h_i_eff = h_i + X @ J_i

    # Calculate and return NLL across individuals
    m = y.shape[0]
    t = 2
    return 1 / m * 1 / t * np.log(1 + np.exp((1 - 2 * y) * h_i_eff)).sum()


def nll_pol(params, X, y):
    J_i = params[:-1]
    h_i = params[-1]

    # Calculate effective local field for each individual
    h_i_eff = h_i + X @ J_i

    # Calculate and return NLL across individuals
    m = y.shape[0]
    t = 2
    return -1 / m * 1 / t * (y * h_i_eff - np.log(2 * np.cosh(h_i_eff))).sum()

In [ ]:
# J_split_oppose.reshape((2, -1))

In [ ]:
# J_split_support.reshape((2, -1))

In [ ]:
# h_split_oppose, h_split_support

In [ ]:
def fit_model_manual(Y, x0, f, use_bounds):
    n = Y.shape[-1]
    h = np.empty(n, dtype=np.float64)
    J = np.empty((n, n), dtype=np.float64)

    bounds = [(0, None)] * n + [(None, None)] if use_bounds else None

    for i in range(n):
        res = minimize(
            f, x0, args=(Y[:, 0], Y[:, 1, i]), bounds=bounds, method="L-BFGS-B"
        )
        h[i] = res.x[-1]
        J[:, i] = res.x[:-1]

    return h, J


def full_nll(h, J, Y, nll):
    _nll = 0.0
    n = Y.shape[-1]

    return np.sum(
        [nll(np.concat((J[:, i], [h[i]])), Y[:, 0], Y[:, 1, i]) for i in range(n)]
    )


def bic(h, J, Y, nll):
    n = Y.shape[0]
    k = h.size + J.size
    return k * np.log(n) + 2 * full_nll(h, J, Y, nll)

In [ ]:
h, J = fit_model_manual(Y_bin, x0=np.zeros(9), f=nll_pol, use_bounds=False)
h_split, J_split = fit_model_manual(
    Y_split, x0=np.zeros(17), f=nll_bin, use_bounds=True
)

In [ ]:
bic(h, J, Y_bin, nll=nll_pol)

In [ ]:
bic(h_split, J_split, Y_split, nll=nll_bin)

In [ ]:
J_split[abs(J_split) < 0.01] = 0.0

In [ ]:
J[:, 7]

In [ ]:
J_split[:, 5].reshape((2, -1))

In [ ]:
J_split[:, 13].reshape((2, -1))

In [ ]:
dataset.schema.get_short_names(kind="measurement")

In [ ]:
np.corrcoef(Y_split[:, 0, 5], Y_split[:, 1, 7])

In [ ]:
dataset.response.collect().head()

In [ ]:
dataset.indices.collect().head()